# Colorization (From grayscale to color)

### Convert color to grayscale

In [ ]:
# import cv2

# def color_to_grayscale(image_path, output_path):
#     # Read the color image
#     img = cv2.imread(image_path)

#     # Convert to grayscale
#     gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

#     # Save and display the result
#     cv2.imwrite(output_path, gray)
#     cv2.imshow("Original Color", img)
#     cv2.imshow("Grayscale", gray)
#     cv2.waitKey(0)
#     cv2.destroyAllWindows()

# # Example usage
# color_to_grayscale("rubik.jpeg", "grayscale.png")

### Convert grayscale to color

In [8]:
import cv2
import numpy as np

def colorize_image_dnn(gray_image_path):

    proto = "colorization_deploy_v2.prototxt"
    model = "colorization_release_v2.caffemodel"
    points = "pts_in_hull.npy"

    net = cv2.dnn.readNetFromCaffe(proto, model)
    pts = np.load(points)

    class8 = net.getLayerId("class8_ab")
    conv8 = net.getLayerId("conv8_313_rh")
    pts = pts.transpose().reshape(2, 313, 1, 1)
    net.getLayer(class8).blobs = [pts.astype("float32")]
    net.getLayer(conv8).blobs = [np.full([1, 313], 2.606, dtype="float32")]

    # Read and process image
    gray = cv2.imread(gray_image_path)
    gray = cv2.cvtColor(gray, cv2.COLOR_BGR2GRAY)
    gray = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)

    h, w = gray.shape[:2]
    img_rgb = (gray[:, :, [2, 1, 0]] / 255.).astype(np.float32)
    img_lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2Lab)
    l = img_lab[:, :, 0]

    l_resized = cv2.resize(l, (224, 224))
    l_resized -= 50
    net.setInput(cv2.dnn.blobFromImage(l_resized))
    ab = net.forward()[0, :, :, :].transpose((1, 2, 0))
    ab = cv2.resize(ab, (w, h))

    lab_out = np.concatenate((l[:, :, np.newaxis], ab), axis=2)
    bgr_out = cv2.cvtColor(lab_out, cv2.COLOR_Lab2BGR)
    bgr_out = np.clip(bgr_out, 0, 1)

    colorized = (bgr_out * 255).astype(np.uint8)
    cv2.imwrite("colorized_result.png", colorized)
    cv2.imshow("Original Grayscale", gray)
    cv2.imshow("Colorized", colorized)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

# Example usage
colorize_image_dnn("grayscale.png")